# Quant Skills — Lesson 7: Pairs Trading (Statistical Arbitrage)

**The classic quant hedge fund strategy.** Find two stocks whose prices are tied together — like Coca-Cola and Pepsi.
When they drift apart, bet on them coming back together: buy the cheap one, short the expensive one.

Because you're long one and short the other, **you don't care which way the market goes** — only whether the gap closes.
That's what makes it "market neutral."

**How to run it:** **Runtime → Run all** (about a minute). Lines you can edit are marked `# <- EDIT`.

## Step 1 — Load the tools

In [ ]:
!pip install yfinance statsmodels -q
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations
from statsmodels.tsa.stattools import adfuller, coint
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
print("Tools loaded.")

## Step 2 — Download candidates

In [ ]:
TICKERS = ["KO", "PEP", "XOM", "CVX", "V", "MA", "HD", "LOW", "JPM", "BAC"]   # <- EDIT
YEARS   = 5         # <- EDIT
TRAIN   = 0.6       # first 60% of history to FIND the pair; last 40% to TRADE it

raw = yf.download(TICKERS, period=f"{YEARS}y", interval="1d", auto_adjust=True, progress=False)
close = raw["Close"][TICKERS].dropna()
split = close.index[int(len(close) * TRAIN)]
train, test = close[close.index < split], close[close.index >= split]
print(f"{len(close)} trading days. Training: up to {split.date()} · Trading: after.")

## Step 3 — Correlation is NOT enough

Two stocks can be highly correlated and still drift apart forever. What pairs traders need is **cointegration**:
the *gap* between them keeps getting pulled back to normal.

The test below fits the relationship `log(A) = α + β × log(B)` on the training data, then runs the **Engle-Granger
cointegration test** on the leftover gap (the **spread**). A **p-value below 0.05** means the gap reliably snaps back.

It also requires the two stocks to genuinely move together day to day (`MIN_CORR`) — a pair needs an economic reason
to be linked, not just a statistical coincidence.

The **half-life** is how many days it typically takes the gap to close halfway — shorter is better for trading.

In [ ]:
def fit_pair(a, b):
    la, lb = np.log(a), np.log(b)
    beta, alpha = np.polyfit(lb, la, 1)
    spread = la - (alpha + beta * lb)
    pval = coint(la, lb)[1]   # Engle-Granger test: uses the stricter critical values a fitted spread needs
    lagged = spread.shift(1).dropna()
    delta = spread.diff().dropna()
    lam = np.polyfit(lagged.values, delta.loc[lagged.index].values, 1)[0]
    half_life = -np.log(2) / lam if lam < 0 else np.inf
    return alpha, beta, pval, half_life

MIN_CORR = 0.6   # <- EDIT: daily-return correlation required to count as a candidate pair

rows = []
for a, b in combinations(TICKERS, 2):
    alpha, beta, pval, hl = fit_pair(train[a], train[b])
    rows.append({"Pair": f"{a}/{b}", "A": a, "B": b, "Correlation": train[a].pct_change().corr(train[b].pct_change()),
                 "p-value": pval, "Half-life (days)": hl, "Beta": beta, "Alpha": alpha})
all_pairs = pd.DataFrame(rows)
pairs = all_pairs[all_pairs["Correlation"] >= MIN_CORR].sort_values("p-value")
print(f"{len(pairs)} of {len(all_pairs)} pairs move together closely enough (correlation ≥ {MIN_CORR}).\n")
if pairs.empty:
    print("None qualify — lower MIN_CORR or add more same-industry candidates in Step 2.")

print(pairs.head(10)[["Pair", "Correlation", "p-value", "Half-life (days)", "Beta"]]
      .round({"Correlation": 2, "p-value": 3, "Half-life (days)": 1, "Beta": 2}).to_string(index=False))
n_pass = (pairs["p-value"] < 0.05).sum()
print(f"\n{n_pass} of {len(pairs)} candidate pairs pass at p < 0.05.")
print(f"Warning: testing {len(pairs)} pairs, about {len(pairs) * 0.05:.1f} would pass by pure chance. "
      f"That's why Step 7 tests the winner on data it has never seen.")

## Step 4 — Pick the pair and see the spread

In [ ]:
PAIR = None   # <- EDIT: e.g. "KO/PEP" — or None to use the best-scoring pair from Step 3

source = all_pairs if PAIR is not None else pairs
row = source.iloc[0] if PAIR is None else all_pairs[all_pairs["Pair"] == PAIR].iloc[0]
A, B, alpha, beta = row["A"], row["B"], row["Alpha"], row["Beta"]
print(f"Trading {A}/{B}: log({A}) = {alpha:.3f} + {beta:.2f} × log({B})   "
      f"·   p-value {row['p-value']:.3f}   ·   half-life {row['Half-life (days)']:.1f} days")

spread_all = np.log(close[A]) - (alpha + beta * np.log(close[B]))
fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax[0].plot(close.index, close[A] / close[A].iloc[0], label=A)
ax[0].plot(close.index, close[B] / close[B].iloc[0], label=B)
ax[0].set_title("Both stocks, scaled to start at 1"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(spread_all.index, spread_all, color="tab:purple")
ax[1].axhline(0, color="black", linewidth=0.8); ax[1].axvline(split, color="red", linestyle="--", label="training ends")
ax[1].set_title("The spread — this is what you actually trade"); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.show()

## Step 5 — The trading rules

Measure how stretched the spread is with a **z-score** — how many standard deviations it sits from its recent average.

| Signal | Action |
|---|---|
| z above **+2** | Spread too wide → **short A, buy B** |
| z below **−2** | Spread too narrow → **buy A, short B** |
| z back inside **±0.5** | Gap closed → **exit, take the profit** |
| z beyond **±4** | Relationship may be breaking → **stop out** |
| Held longer than 3× the half-life | Not reverting → **time stop** |

In [ ]:
LOOKBACK = 60     # <- EDIT: days used for the z-score's average and std dev
ENTRY_Z  = 2.0    # <- EDIT
EXIT_Z   = 0.5    # <- EDIT
STOP_Z   = 4.0    # <- EDIT
COST     = 0.001  # 0.10% per side, both legs — commissions, slippage and short borrow
MAX_HOLD = int(max(5, 3 * row["Half-life (days)"])) if np.isfinite(row["Half-life (days)"]) else 60

def backtest_pair(prices_a, prices_b, beta):
    spread = np.log(prices_a) - beta * np.log(prices_b)
    z = (spread - spread.rolling(LOOKBACK).mean()) / spread.rolling(LOOKBACK).std()
    ra, rb = prices_a.pct_change(), prices_b.pct_change()
    gross = 1 + abs(beta)
    pos, held, entry_i = 0, 0, None
    positions, trades = [], []
    for i in range(len(z)):
        zi = z.iloc[i]
        new = pos
        if np.isnan(zi):
            new = 0
        elif pos == 0:
            if zi > ENTRY_Z:  new = -1
            elif zi < -ENTRY_Z: new = 1
        else:
            held += 1
            if abs(zi) < EXIT_Z or abs(zi) > STOP_Z or held > MAX_HOLD:
                new = 0
        if new != pos:
            if pos == 0:
                entry_i, held = i, 0
            elif entry_i is not None:
                reason = "converged" if abs(zi) < EXIT_Z else ("stop" if abs(zi) > STOP_Z else "time stop")
                trades.append({"entry": z.index[entry_i], "exit": z.index[i], "side": "long spread" if pos == 1 else "short spread",
                               "days": i - entry_i, "reason": reason})
            pos = new
        positions.append(pos)
    pos_s = pd.Series(positions, index=z.index)
    daily = pos_s.shift(1) * (ra - beta * rb) / gross
    daily -= pos_s.diff().abs().fillna(0) * COST
    daily = daily.fillna(0)
    tr = pd.DataFrame(trades)
    if not tr.empty:
        tr["return"] = [daily.loc[(daily.index > r.entry) & (daily.index <= r.exit)].add(1).prod() - 1 for r in tr.itertuples()]
    return daily, z, pos_s, tr

def stats(daily, label):
    eq = (1 + daily).cumprod()
    yrs = len(daily) / 252
    cagr = eq.iloc[-1] ** (1 / yrs) - 1 if yrs > 0 else np.nan
    vol = daily.std() * np.sqrt(252)
    sharpe = daily.mean() / daily.std() * np.sqrt(252) if daily.std() > 0 else np.nan
    dd = (eq / eq.cummax() - 1).min()
    print(f"{label}: annual return {cagr:.1%}  ·  volatility {vol:.1%}  ·  Sharpe {sharpe:.2f}  ·  worst drawdown {dd:.1%}")
    return eq

print(f"Rules set. Time stop: {MAX_HOLD} days.")

## Step 6 — Results on the training period (where the pair was found)

In [ ]:
d_tr, z_tr, p_tr, t_tr = backtest_pair(train[A], train[B], beta)
eq_tr = stats(d_tr, "Training period")
if not t_tr.empty:
    print(f"Trades: {len(t_tr)}  ·  win rate {(t_tr['return'] > 0).mean():.0%}  ·  "
          f"average trade {t_tr['return'].mean():+.2%}  ·  average hold {t_tr['days'].mean():.0f} days")
    print(t_tr["reason"].value_counts().to_string())

## Step 7 — The real test: data it has never seen

Same pair, same β from the training period, traded on the last 40% of history. **This is the number that counts.**
If the relationship was real, it keeps working. If it was luck from scanning many pairs, it falls apart here.

In [ ]:
d_te, z_te, p_te, t_te = backtest_pair(test[A], test[B], beta)
eq_te = stats(d_te, "Unseen period ")
if not t_te.empty:
    print(f"Trades: {len(t_te)}  ·  win rate {(t_te['return'] > 0).mean():.0%}  ·  "
          f"average trade {t_te['return'].mean():+.2%}  ·  average hold {t_te['days'].mean():.0f} days")
    print(t_te["reason"].value_counts().to_string())

_, _, p_now, hl_now = fit_pair(test[A], test[B])
print(f"\nIs the pair still cointegrated in the unseen period? p-value {p_now:.3f} "
      f"({'yes' if p_now < 0.05 else 'NO — the relationship weakened'})")

fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax[0].plot(z_te.index, z_te, color="tab:purple", linewidth=1)
for lvl, c in ((ENTRY_Z, "red"), (-ENTRY_Z, "green"), (EXIT_Z, "gray"), (-EXIT_Z, "gray")):
    ax[0].axhline(lvl, color=c, linestyle="--", linewidth=0.8)
ax[0].fill_between(z_te.index, -10, 10, where=p_te != 0, color="orange", alpha=0.12, label="in a trade")
ax[0].set_ylim(-5, 5); ax[0].set_title(f"{A}/{B} z-score — unseen period"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(eq_te.index, eq_te, linewidth=1.5); ax[1].axhline(1, color="black", linewidth=0.8)
ax[1].set_title("Growth of $1"); ax[1].grid(alpha=0.3)
plt.show()

## What this teaches — and the traps

| Lesson | Why it matters |
|---|---|
| **Market neutral** | You're long one, short the other — a market crash hits both legs, so the spread is what matters |
| **Correlation ≠ cointegration** | Highly correlated stocks can still drift apart for years |
| **Multiple testing** | Scan enough pairs and some will "pass" by luck — Step 3 counts how many |
| **Relationships break** | Mergers, new products, regulation can permanently change a pair. The ±4 stop exists for this |
| **Costs are bigger here** | Two legs, plus the cost of borrowing shares to short — edges are thin |
| **It needs a margin account** | Shorting stock requires margin approval |

## Your exercises

1. **Step 4:** set `PAIR = "KO/PEP"` and run all. Is the classic textbook pair still cointegrated?
2. **Step 7:** does the best pair from Step 3 still pass on the unseen data — or was it luck?
3. **Step 5:** try `ENTRY_Z = 1.5` and `2.5`. More trades or better trades?
4. **Step 2:** add your own candidates — two stocks in the same industry — and see if any genuinely pair up.

**This lesson plus Lessons 5 and 6 are your GitHub portfolio.** A project that honestly reports a strategy breaking
out-of-sample shows *more* skill than one that claims it always works.